### 6.1 Chargement des données

Nous allons commencer par charger le dataset obtenu après l'analyse exploratoire.

Cette étape permet de travailler dans un nouveau notebook dédié au **Feature Engineering** et à la préparation des données pour la modélisation.


In [27]:
import pandas as pd

df = pd.read_csv("../data/cleaned_data.csv")

df.head()

,Delivery_person_Age,Delivery_person_Ratings,Restaurant_latitude,Restaurant_longitude,Delivery_location_latitude,Delivery_location_longitude,Order_Date,Weatherconditions,Road_traffic_density,Type_of_vehicle,multiple_deliveries,Time_taken(min),distance,order_hour,pickup_hour,pickup_delay
0,37.0,4.9,22.745049,75.892471,22.765049,75.912471,2022-03-19,Sunny,High,motorcycle,0.0,24,3.020737,11.0,11,15.0
1,34.0,4.5,12.913041,77.683237,13.043041,77.813237,2022-03-25,Stormy,Jam,scooter,1.0,33,20.143737,19.0,19,5.0
2,23.0,4.4,12.914264,77.678400,12.924264,77.688400,2022-03-19,Sandstorms,Low,motorcycle,1.0,26,1.549693,8.0,8,15.0
3,38.0,4.7,11.003669,76.976494,11.053669,77.026494,2022-04-05,Sunny,Medium,motorcycle,1.0,21,7.774497,18.0,18,10.0
4,32.0,4.6,12.972793,80.249982,13.012793,80.289982,2022-03-26,Cloudy,High,scooter,1.0,30,6.197898,13.0,13,15.0


In [28]:
df.shape


(41522, 16)

In [29]:
df.info()
df.isna().sum()

<class 'pandas.DataFrame'>
RangeIndex: 41522 entries, 0 to 41521
Data columns (total 16 columns):
 #   Column                       Non-Null Count  Dtype  
---  ------                       --------------  -----  
 0   Delivery_person_Age          41522 non-null  float64
 1   Delivery_person_Ratings      41522 non-null  float64
 2   Restaurant_latitude          41522 non-null  float64
 3   Restaurant_longitude         41522 non-null  float64
 4   Delivery_location_latitude   41522 non-null  float64
 5   Delivery_location_longitude  41522 non-null  float64
 6   Order_Date                   41522 non-null  str    
 7   Weatherconditions            41522 non-null  str    
 8   Road_traffic_density         41522 non-null  str    
 9   Type_of_vehicle              41522 non-null  str    
 10  multiple_deliveries          41522 non-null  float64
 11  Time_taken(min)              41522 non-null  int64  
 12  distance                     41522 non-null  float64
 13  order_hour                 

Delivery_person_Age            0
Delivery_person_Ratings        0
Restaurant_latitude            0
Restaurant_longitude           0
Delivery_location_latitude     0
Delivery_location_longitude    0
Order_Date                     0
Weatherconditions              0
Road_traffic_density           0
Type_of_vehicle                0
multiple_deliveries            0
Time_taken(min)                0
distance                       0
order_hour                     0
pickup_hour                    0
pickup_delay                   0
dtype: int64

### Extracting the day of the week

The `Order_Date` column contains dates in `YYYY-MM-DD` format. We convert it to datetime and extract the day of the week.

This feature can help the model capture differences between weekdays and weekends.


In [30]:
df["Order_Date"] = pd.to_datetime(
    df["Order_Date"],
    format="%Y-%m-%d"
)

df["day_of_week"] = df["Order_Date"].dt.day_name()

In [31]:
print(df[["Order_Date", "day_of_week"]].head(10))

  Order_Date day_of_week
0 2022-03-19    Saturday
1 2022-03-25      Friday
2 2022-03-19    Saturday
3 2022-04-05     Tuesday
4 2022-03-26    Saturday
5 2022-03-11      Friday
6 2022-03-04      Friday
7 2022-03-14      Monday
8 2022-03-20      Sunday
9 2022-02-12    Saturday


### Creating a weekend indicator

We create `is_weekend` from `Order_Date` to indicate whether an order was placed on a weekend.

`0` represents a weekday and `1` represents Saturday or Sunday. This can help the model capture possible differences in delivery behavior between weekdays and weekends.


In [32]:
df["is_weekend"] = (
    df["Order_Date"].dt.dayofweek >= 5
).astype(int) # cooonvert True/False to 1/0

In [33]:
print(df[["Order_Date", "day_of_week", "is_weekend"]].head(10))

  Order_Date day_of_week  is_weekend
0 2022-03-19    Saturday           1
1 2022-03-25      Friday           0
2 2022-03-19    Saturday           1
3 2022-04-05     Tuesday           0
4 2022-03-26    Saturday           1
5 2022-03-11      Friday           0
6 2022-03-04      Friday           0
7 2022-03-14      Monday           0
8 2022-03-20      Sunday           1
9 2022-02-12    Saturday           1


### Extracting `month`

We extract the month from `Order_Date` to capture possible seasonal patterns that may affect delivery time.



In [34]:
df["month"] = df["Order_Date"].dt.month
print(df[["Order_Date", "month"]].head(10))

  Order_Date  month
0 2022-03-19      3
1 2022-03-25      3
2 2022-03-19      3
3 2022-04-05      4
4 2022-03-26      3
5 2022-03-11      3
6 2022-03-04      3
7 2022-03-14      3
8 2022-03-20      3
9 2022-02-12      2


In [35]:
print(df["month"].value_counts().sort_index())

month
2     5819
3    29773
4     5930
Name: count, dtype: int64


In [36]:
print(df[[
    "Order_Date",
    "day_of_week",
    "is_weekend",
    "month"
]].head(10))

  Order_Date day_of_week  is_weekend  month
0 2022-03-19    Saturday           1      3
1 2022-03-25      Friday           0      3
2 2022-03-19    Saturday           1      3
3 2022-04-05     Tuesday           0      4
4 2022-03-26    Saturday           1      3
5 2022-03-11      Friday           0      3
6 2022-03-04      Friday           0      3
7 2022-03-14      Monday           0      3
8 2022-03-20      Sunday           1      3
9 2022-02-12    Saturday           1      2


In [37]:
print(df[[
    "day_of_week",
    "is_weekend",
    "month"
]].info())

<class 'pandas.DataFrame'>
RangeIndex: 41522 entries, 0 to 41521
Data columns (total 3 columns):
 #   Column       Non-Null Count  Dtype
---  ------       --------------  -----
 0   day_of_week  41522 non-null  str  
 1   is_weekend   41522 non-null  int64
 2   month        41522 non-null  int32
dtypes: int32(1), int64(1), str(1)
memory usage: 1.1 MB
None


In [38]:
print(df["day_of_week"].value_counts())
print(df["is_weekend"].value_counts())
print(df["month"].value_counts().sort_index())

day_of_week
Wednesday    6499
Friday       6325
Tuesday      5812
Thursday     5785
Saturday     5750
Sunday       5685
Monday       5666
Name: count, dtype: int64
is_weekend
0    30087
1    11435
Name: count, dtype: int64
month
2     5819
3    29773
4     5930
Name: count, dtype: int64


### Feature Engineering Summary

Feature engineering was used to transform raw information into features that can be more useful for predicting delivery time.

* `order_hour` was extracted from the order time.
* `pickup_hour` was extracted from the pickup time.
* `distance` was calculated from restaurant and delivery coordinates.
* `day_of_week` was extracted from `Order_Date`.
* `is_weekend` indicates whether the order was placed on Saturday or Sunday.
* `month` was extracted from `Order_Date` to capture possible seasonal patterns.


In [ ]:
#df.to_csv("../data/feature_engineered_data.csv", index=False)

## Pickup Delay

The pickup delay represents the time between placing an order and the order being picked up.

This feature gives the model more precise information than using only the order and pickup hours.

In [40]:
raw_df = pd.read_csv("../data/dataset_food_delivery.csv")

print(raw_df.shape)
print(raw_df.columns.tolist())

(45593, 20)
['ID', 'Delivery_person_ID', 'Delivery_person_Age', 'Delivery_person_Ratings', 'Restaurant_latitude', 'Restaurant_longitude', 'Delivery_location_latitude', 'Delivery_location_longitude', 'Order_Date', 'Time_Orderd', 'Time_Order_picked', 'Weatherconditions', 'Road_traffic_density', 'Vehicle_condition', 'Type_of_order', 'Type_of_vehicle', 'multiple_deliveries', 'Festival', 'City', 'Time_taken(min)']


In [41]:
# Convert order and pickup times to datetime
raw_df["Time_Orderd"] = pd.to_datetime(
    raw_df["Time_Orderd"],
    format="%H:%M:%S",
    errors="coerce"
)

raw_df["Time_Order_picked"] = pd.to_datetime(
    raw_df["Time_Order_picked"],
    format="%H:%M:%S",
    errors="coerce"
)

# Calculate pickup delay in minutes
raw_df["pickup_delay"] = (
    raw_df["Time_Order_picked"] - raw_df["Time_Orderd"]
).dt.total_seconds() / 60

In [42]:
raw_df[
    ["Time_Orderd", "Time_Order_picked", "pickup_delay"]
].head(10)

,Time_Orderd,Time_Order_picked,pickup_delay
0,1900-01-01 11:30:00,1900-01-01 11:45:00,15.0
1,1900-01-01 19:45:00,1900-01-01 19:50:00,5.0
2,1900-01-01 08:30:00,1900-01-01 08:45:00,15.0
3,1900-01-01 18:00:00,1900-01-01 18:10:00,10.0
4,1900-01-01 13:30:00,1900-01-01 13:45:00,15.0
5,1900-01-01 21:20:00,1900-01-01 21:30:00,10.0
6,1900-01-01 19:15:00,1900-01-01 19:30:00,15.0
7,1900-01-01 17:25:00,1900-01-01 17:30:00,5.0
8,1900-01-01 20:55:00,1900-01-01 21:05:00,10.0
9,1900-01-01 21:55:00,1900-01-01 22:10:00,15.0


In [44]:
cleaned_df = pd.read_csv("../data/cleaned_data.csv")

print(cleaned_df.columns.tolist())
print(cleaned_df.shape)

['Delivery_person_Age', 'Delivery_person_Ratings', 'Restaurant_latitude', 'Restaurant_longitude', 'Delivery_location_latitude', 'Delivery_location_longitude', 'Order_Date', 'Weatherconditions', 'Road_traffic_density', 'Type_of_vehicle', 'multiple_deliveries', 'Time_taken(min)', 'distance', 'order_hour', 'pickup_hour', 'pickup_delay']
(41522, 16)


In [45]:
df["Order_Date"] = pd.to_datetime(df["Order_Date"])

df["day_of_week"] = df["Order_Date"].dt.day_name()
df["is_weekend"] = (df["Order_Date"].dt.dayofweek >= 5).astype(int)
df["month"] = df["Order_Date"].dt.month

In [46]:
df.shape

(41522, 19)

In [47]:
df.columns

Index(['Delivery_person_Age', 'Delivery_person_Ratings', 'Restaurant_latitude',
       'Restaurant_longitude', 'Delivery_location_latitude',
       'Delivery_location_longitude', 'Order_Date', 'Weatherconditions',
       'Road_traffic_density', 'Type_of_vehicle', 'multiple_deliveries',
       'Time_taken(min)', 'distance', 'order_hour', 'pickup_hour',
       'pickup_delay', 'day_of_week', 'is_weekend', 'month'],
      dtype='str')

In [48]:
df.to_csv(
    "../data/feature_engineered_data.csv",
    index=False
)

In [49]:
check_df = pd.read_csv("../data/feature_engineered_data.csv")

print(check_df.shape)
print(check_df.columns)

(41522, 19)
Index(['Delivery_person_Age', 'Delivery_person_Ratings', 'Restaurant_latitude',
       'Restaurant_longitude', 'Delivery_location_latitude',
       'Delivery_location_longitude', 'Order_Date', 'Weatherconditions',
       'Road_traffic_density', 'Type_of_vehicle', 'multiple_deliveries',
       'Time_taken(min)', 'distance', 'order_hour', 'pickup_hour',
       'pickup_delay', 'day_of_week', 'is_weekend', 'month'],
      dtype='str')
